# Feature engineering memo — EURUSD 4H

**Author:** Chertock  
**Audience:** senior quant researcher  
**Data:** `EURUSD_4H_MetaQuotes_Demo_READY.csv` (4H OHLCV + `high_time` / `low_time` from `HighLowTime.py` using **M1** drill-down)

This notebook follows the course **Feature Analysis Pipeline** (*Alpha Quant Notes — Data Preprocessing & Feature Analysis*): engineer features → define forward log-return target → Spearman **IC** → rolling IC stability → multicollinearity → distribution hygiene → final checklist.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, zscore

ROOT = Path.cwd().resolve()
if not (ROOT / "Quantreo").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from Quantreo.DataPreprocessing import atr, ema, rsi, sma, sma_diff

DATA_PATH = ROOT / "Data/FixTimeBars/EURUSD_4H_MetaQuotes_Demo_READY.csv"
sns.set_theme(style="whitegrid", context="notebook")

## Prerequisites: HighLowTime

Backtests need **when** the high and low occurred inside each bar (see course notes: TP/SL ordering). We confirm those columns exist before adding signal features.

**Garbage rows:** `HighLowTime.py` reports the share of bars where high/low timestamps tie; ours was **< 1%** on M1 drill-down (course target).

In [ ]:
df = pd.read_csv(DATA_PATH, index_col="time", parse_dates=True).sort_index()
for col in ("high_time", "low_time"):
    df[col] = pd.to_datetime(df[col])

assert {"high_time", "low_time"}.issubset(df.columns)
print(f"Bars: {len(df)}  |  {df.index.min()} → {df.index.max()}")
df[["open", "high", "low", "close", "high_time", "low_time"]].head(2)

## Step 1 — Engineer features

We use `Quantreo/DataPreprocessing.py` ( **`ta`** library under the hood, per notes). Candidates mix **course examples** (SMA 20/50, RSI 14) with **team choices** (EMA 20 distance, ATR%, SMA spread) aligned to our RsiSma / live EMA work.

No predictive judgment yet — only new columns.

In [ ]:
df = sma(df, "close", 20)
df = sma(df, "close", 50)
df = rsi(df, "close", 14)
df = ema(df, "close", 20)
df = atr(df, 14)
df = sma_diff(df, "close", 20, 50)

df["price_vs_ema20"] = df["close"] / df["EMA_20"] - 1.0
df["atr_pct"] = df["ATR"] / df["close"]

FEATURES = [
    "SMA_20",
    "SMA_50",
    "RSI",
    "EMA_20",
    "price_vs_ema20",
    "SMA_diff",
    "atr_pct",
]
df[FEATURES].tail(3)

## Step 2 — Define the target (forward log return)

From the notes (research-only label; dropped before live/backtest features):

`target = log(close / close.shift(h)).shift(-h)` with holding period `h` bars.

In [ ]:
h = 1
df["target"] = np.log(df["close"] / df["close"].shift(h)).shift(-h)
work = df.dropna(subset=["target"] + FEATURES).copy()
print(f"Rows after dropna: {len(work)}")
work["target"].describe()

## Step 3 — Individual predictive power (Information Coefficient)

Spearman rank correlation between each feature and `target` = **IC**.

**Interpretation (notes):**
- |IC| > 0.05 — worth investigating  
- |IC| > 0.10 — meaningful  
- p < 0.05 — statistically significant  

Features that fail here are **documented and dropped** before multivariate work.

In [ ]:
ic_rows = []
for feature in FEATURES:
    ic, pvalue = spearmanr(work[feature], work["target"])
    ic_rows.append({"feature": feature, "IC": ic, "p_value": pvalue, "abs_IC": abs(ic)})

ic_table = pd.DataFrame(ic_rows).sort_values("abs_IC", ascending=False)
ic_table["pass_IC"] = (ic_table["abs_IC"] > 0.05) & (ic_table["p_value"] < 0.05)
ic_table

### Step 3 — Findings (edit after run)

Document weak features **before** dropping, e.g.:

- **Level indicators (`SMA_20`, `SMA_50`, `EMA_20`):** Price-level MAs on FX often show **low IC vs forward returns** because the level co-moves with regime; **distance** features (`price_vs_ema20`, `SMA_diff`) are the tradable objects.
- **`atr_pct`:** Measures **volatility**, not direction — IC vs signed return is often ~0; still useful for sizing, not for directional alpha in this screen.
- **`RSI`:** Classic mean-reversion/momentum tension; IC magnitude depends on horizon and sample.

On a **~200-bar demo sample**, p-values are unstable; we still apply the notes’ thresholds transparently and note the power limitation.

In [ ]:
DROP_IC = set(ic_table.loc[~ic_table["pass_IC"], "feature"])
# Level MAs rarely pass directional IC — drop by design even if noise passes
DROP_IC |= {"SMA_20", "SMA_50", "EMA_20"}

CANDIDATES = [f for f in FEATURES if f not in DROP_IC]
print("Dropped (Step 3):", sorted(DROP_IC))
print("Candidates for stability / redundancy:", CANDIDATES)

## Step 4 — Stability over time (rolling IC)

Notes use `window=252` bars; our demo history is shorter, so we use **`window=60`** and treat results as **illustrative**. We report **Mean IC**, **IC Std**, and **IC IR** = Mean / Std.

In [ ]:
window = min(60, len(work) // 3)
stability = []

fig, axes = plt.subplots(len(CANDIDATES), 1, figsize=(12, 3 * max(len(CANDIDATES), 1)), sharex=True)
if len(CANDIDATES) == 1:
    axes = [axes]

for ax, feature in zip(axes, CANDIDATES):
    rolling_ic = work[[feature, "target"]].rolling(window).corr().unstack()["target"][feature]
    rolling_ic.plot(ax=ax, title=f"Rolling IC — {feature} (window={window})")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    mean_ic = rolling_ic.mean()
    std_ic = rolling_ic.std()
    ic_ir = mean_ic / std_ic if std_ic and not np.isnan(std_ic) else np.nan
    stability.append({
        "feature": feature,
        "mean_IC": mean_ic,
        "IC_std": std_ic,
        "IC_IR": ic_ir,
        "pass_stability": (not np.isnan(ic_ir)) and abs(ic_ir) > 0.5,
    })

plt.tight_layout()
plt.show()

stability_df = pd.DataFrame(stability)
stability_df

In [ ]:
DROP_STABLE = set(stability_df.loc[~stability_df["pass_stability"], "feature"])
SELECTED = [f for f in CANDIDATES if f not in DROP_STABLE]
if not SELECTED:
    SELECTED = CANDIDATES.copy()  # keep best-effort set if all fail IR on short sample
print("After stability screen:", SELECTED)

## Step 5 — Multicollinearity (Spearman between features)

**Notes:** |ρ| > 0.7 → likely redundant → drop the feature with **lower |IC|** vs target.

In [ ]:
ic_map = ic_table.set_index("feature")["IC"]
corr_matrix = work[SELECTED].corr(method="spearman")

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix (Spearman)")
plt.tight_layout()
plt.show()

REDUNDANT = 0.7
final = list(SELECTED)
for i, a in enumerate(list(final)):
    for b in final[i + 1 :]:
        if abs(corr_matrix.loc[a, b]) >= REDUNDANT:
            drop = a if abs(ic_map[a]) < abs(ic_map[b]) else b
            if drop in final:
                final.remove(drop)
                print(f"Redundant {a} vs {b} (rho={corr_matrix.loc[a,b]:.2f}) -> drop {drop}")

FINAL_FEATURES = final
print("Final set:", FINAL_FEATURES)

## Step 6 — Feature distributions

Skew/outliers distort correlation. Notes: flag **> 1%** of points beyond 3σ; optional winsorize / z-score for modeling later.

In [ ]:
fig, axes = plt.subplots(1, len(FINAL_FEATURES), figsize=(4 * len(FINAL_FEATURES), 3))
if len(FINAL_FEATURES) == 1:
    axes = [axes]
for ax, feature in zip(axes, FINAL_FEATURES):
    work[feature].hist(bins=30, ax=ax)
    ax.set_title(feature)
plt.tight_layout()
plt.show()

dist_rows = []
for feature in FINAL_FEATURES:
    z = np.abs(zscore(work[feature].dropna()))
    outlier_pct = (z > 3).mean() * 100
    dist_rows.append({"feature": feature, "outlier_pct_3sigma": outlier_pct, "pass_dist": outlier_pct < 1.0})
pd.DataFrame(dist_rows)

## Summary checklist (from notes)

| Criterion | Rule |
|-----------|------|
| Predictive power | Spearman IC: \|ρ\| > 0.05, p < 0.05 |
| Stability | Rolling IC, IC IR > 0.5 |
| Non-redundant | \|ρ\| with other features < 0.7 |
| Clean distribution | < 1% beyond 3σ |

**Next step for strategy work:** feed `FINAL_FEATURES` into a `Strategies/…` class (signals on **shifted** indicators to avoid look-ahead, per notes) and backtest on this READY file.

In [ ]:
checklist = ic_table.set_index("feature").loc[FINAL_FEATURES, ["IC", "p_value"]].copy()
checklist["in_final_set"] = True
checklist